# 02. Baseline pipeline: contract, split и leakage-control

**Цель.** Построить проверочную постановку из `raw_data/`: окна наблюдения, строгую разметку до explicit anchor, split и baseline. Ноутбук не хранит тяжелый парсинг внутри себя, а вызывает воспроизводимый модуль `dashboard.final_data_pipeline`.


In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dashboard.final_data_pipeline import PROJECT_ROOT, SEED, artifact_paths, run_pipeline

random.seed(SEED)
np.random.seed(SEED)

RAW_DATA = PROJECT_ROOT / "raw_data"
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
PATHS = artifact_paths(ARTIFACT_DIR)

print(f"raw_data: {RAW_DATA}")
print(f"artifact_dir: {ARTIFACT_DIR}")


## 1. Сборка strict/intensity contract

Contract строится напрямую из `raw_data`: raw logs, KPI-файлы, временная сетка `10s`, macro-контекст логов `60s` и `gap10` до фактического события. Для failure-run используются только explicit anchors из логов; окна, попадающие слишком близко к anchor, исключаются из обучения.


In [ ]:
report = run_pipeline(raw_data=RAW_DATA, artifact_dir=ARTIFACT_DIR)
print(json.dumps(report["written_files"], ensure_ascii=False, indent=2))


**Локальный вывод.** После этой ячейки весь downstream-контур получает один канонический вход: `dashboard/artifacts/contract/strict_intensity_contract.csv.gz`. Старые исследовательские артефакты не являются входом финальной системы.


## 2. Split и целевые признаки

Проверяем, что split не смешивает одинаковые окна одного run и что обе целевые шкалы не вырождены. Binary-цели нужны для строгой проверки, intensity-цели — для обучения нарастанию риска по времени до события.


In [ ]:
contract = pd.read_csv(PATHS.contract_path, parse_dates=["window_ts", "window_end"], low_memory=False)
split_summary = (
    contract.groupby("split")
    .agg(
        rows=("run_id", "size"),
        runs=("run_id", "nunique"),
        workloads=("workload", "nunique"),
        warning_share=("y_warning", "mean"),
        risk_share=("y_risk", "mean"),
        both_modalities=("has_both_modalities", "mean"),
    )
    .reset_index()
)
display(split_summary)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(split_summary["split"], split_summary["rows"], color="#2563eb")
ax.set_title("Размеры split в strict/intensity contract")
ax.set_ylabel("Кол-во окон")
ax.set_xlabel("Split")
plt.tight_layout()
plt.show()


**Что показано.** Распределение окон, запусков и целевых долей по split.  
**Локальный вывод.** Финальные метрики можно интерпретировать только вместе с долей positive-окон и покрытием обеих модальностей; иначе высокий score может оказаться следствием дисбаланса или близости к anchor.


## 3. Baseline как нижняя граница

Baseline не является финальной системой. Он нужен, чтобы проверить, есть ли в contract обучаемый сигнал и как ведут себя log-only, KPI-only, availability-only и простой fused вариант на том же split.


In [ ]:
baseline = pd.read_csv(PATHS.baseline_comparison_path)
compact = baseline.sort_values(["target", "average_precision"], ascending=[True, False])
display(compact)

fig, ax = plt.subplots(figsize=(8, 3.8))
for target, part in compact.groupby("target"):
    ax.plot(part["baseline"], part["average_precision"], marker="o", label=target)
ax.set_title("Baseline: Average Precision на test")
ax.set_ylabel("Average Precision")
ax.set_xlabel("Baseline")
ax.tick_params(axis="x", rotation=20)
ax.legend()
plt.tight_layout()
plt.show()


**Локальный вывод.** Baseline фиксирует нижнюю границу и риск логовой доминации. Финальная модель в `03` должна обучаться как единая мультимодальная fusion-система, а не как две независимые модели, и затем отдавать score в alert-layer дашборда.


**Handoff в 03.** Передается contract `dashboard/artifacts/contract/strict_intensity_contract.csv.gz` и baseline-прогнозы `dashboard/artifacts/reports/baseline_predictions.csv.gz`. Следующий ноутбук обучает финальную мультимодальную модель и экспортирует replay bundle для dashboard.
